# ACP Showcase — Drei Szenarien in einem Notebook

Begleit-Notebook zu [`docs/features/acp.md`](../docs/features/acp.md). Demonstriert das **Agent Communication Protocol** (ACP) anhand dreier realer Setups:

| # | Szenario | Worum es geht |
|---|----------|--------------|
| 1 | **3-Peer-Demo** | Orchestrator (8800) delegiert per `call_acp_agent` an Researcher (8801) + Coder (8802). Mehrstufige Mission; nutzt die `showcase_*` Profile aus `src/taskforce/configs/`. |
| 2 | **Butler-Style Single-Delegation** | Chat-artiger Agent (`showcase_butler`) ruft den Researcher genau einmal auf und synthetisiert die Antwort lokal — die praktische "Butler-on-Telegram"-Variante. |
| 3 | **Message Bus über ACP** | Zwei loopback `AcpRuntime`s, einer `publish`, der andere `subscribe`. Reiner Protocol-Layer-Test ohne LLM, läuft in-process im Notebook. |

**Voraussetzungen**

- `uv sync --extra acp` (acp-sdk 1.0.x + uvicorn<0.36 sind gepinnt)
- LLM-Credentials in `.env` (Azure OpenAI per Default in `src/taskforce/configs/llm_config.yaml`)
- Freie Ports 8800–8802 für Szenario 1 + 2
- Notebook im Repo-Root starten (oder in `notebooks/` — wir wechseln gleich automatisch)
- **Windows:** `PYTHONIOENCODING=utf-8` wird im Setup-Cell automatisch gesetzt — sonst crasht Rich's Box-Drawing-Output mit `UnicodeEncodeError` unter der default cp1252-Codepage.

**Laufzeit-Erwartung** (mit `azure/gpt-5.4-mini`)

- Szenario 1: ~3–6 Minuten (Mehrstufige Mission, drei LLM-Agenten)
- Szenario 2: ~1–3 Minuten (Eine Delegation)
- Szenario 3: <15 Sekunden (kein LLM)

## 0. Setup

Repo-Root finden, `.env` laden, Encoding für Windows fixen, `acp-sdk` prüfen und ein paar Helper-Funktionen für Peer-Management definieren.

In [ ]:
import os, sys, socket, subprocess, time, json, signal
from pathlib import Path

# Repo-Root finden (Notebook kann unter notebooks/ oder im Repo-Root liegen).
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print("cwd:", REPO_ROOT)

# .env optional laden.
try:
    from dotenv import load_dotenv
    if (REPO_ROOT / ".env").exists():
        load_dotenv(REPO_ROOT / ".env")
        print(".env geladen")
except ImportError:
    pass

# UTF-8 fuer alle Subprozesse erzwingen (Windows-Fix).
os.environ["PYTHONIOENCODING"] = "utf-8"

# Welche LLM-Keys sind da?
for k in ("OPENAI_API_KEY", "AZURE_API_KEY", "AZURE_API_BASE", "AZURE_API_VERSION", "ANTHROPIC_API_KEY"):
    print(f"  {k:24s} {'OK' if os.getenv(k) else '-'}")

# acp-sdk vorhanden?
try:
    import acp_sdk  # noqa: F401
    print("acp-sdk OK")
except ImportError:
    print("!! acp-sdk fehlt — bitte `uv sync --extra acp` ausfuehren")

In [ ]:
# ---- Hilfsfunktionen fuer Peer-Management -------------------------------

LOG_DIR = REPO_ROOT / "examples" / "acp_showcase" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

def is_port_open(port: int, host: str = "127.0.0.1", timeout: float = 0.5) -> bool:
    """True wenn TCP-Connect auf (host, port) klappt."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

def wait_for_port(port: int, attempts: int = 120, sleep: float = 0.5) -> bool:
    """Wartet bis port belegt ist; True bei Erfolg, False bei Timeout (~60s default)."""
    for _ in range(attempts):
        if is_port_open(port):
            return True
        time.sleep(sleep)
    return False

def start_peer(profile: str, log_basename: str) -> subprocess.Popen:
    """Startet `taskforce acp start --profile <profile>` als Hintergrundprozess.

    stdout/stderr werden in examples/acp_showcase/logs/<basename>.log/.err umgeleitet.
    """
    out = open(LOG_DIR / f"{log_basename}.log", "w", encoding="utf-8")
    err = open(LOG_DIR / f"{log_basename}.err", "w", encoding="utf-8")
    proc = subprocess.Popen(
        ["uv", "run", "taskforce", "acp", "start", "--profile", profile],
        cwd=str(REPO_ROOT),
        stdout=out,
        stderr=err,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    return proc

def kill_proc_tree(proc: subprocess.Popen) -> None:
    """Killt den Prozessbaum (uv -> python -> uvicorn). Best-effort."""
    if proc.poll() is not None:
        return
    if sys.platform == "win32":
        subprocess.run(
            ["taskkill", "/PID", str(proc.pid), "/T", "/F"],
            capture_output=True,
        )
    else:
        try:
            proc.terminate()
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()

def free_ports(ports: list[int]) -> None:
    """Berichtet welche Ports noch belegt sind (zum manuellen Aufraeumen)."""
    for p in ports:
        state = "BELEGT" if is_port_open(p) else "frei"
        print(f"  :{p}  {state}")

free_ports([8800, 8801, 8802])

## 1. Szenario 1 — Drei Peers, eine Mission

```
                     +------------------------+
  user mission ----> |  orchestrator  :8800   |
                     |  plan_and_execute      |
                     +-+--------------------+-+
                       | call_acp_agent     | call_acp_agent
                       v                    v
           +---------------------+  +---------------------+
           |  researcher  :8801  |  |    coder    :8802   |
           |  web_search, fetch  |  |  python, file_write |
           +---------------------+  +---------------------+
```

Der Orchestrator hat **kein** Web-Tool, **kein** Python-Tool. Sein `system_prompt` zwingt ihn, jede Recherche an `researcher` und jede Code-Aufgabe an `coder` zu delegieren. Ohne diesen Prompt halluzinieren kleine Modelle die Recherche aus Trainingsdaten — das war einer der Bugs, der dieses Notebook produziert hat.

Die drei Profile liegen in `src/taskforce/configs/showcase_orchestrator.yaml`, `showcase_researcher.yaml`, `showcase_coder.yaml`.

In [ ]:
# Falls aus einem vorherigen Notebook-Lauf noch was haengt: hier sauber starten.
scenario1_peers: list[subprocess.Popen] = []

scenario1_peers.append(start_peer("showcase_researcher", "nb_s1_researcher"))
print("researcher gestartet, warte auf :8801 ...")
assert wait_for_port(8801), "researcher kam nicht hoch — siehe examples/acp_showcase/logs/nb_s1_researcher.err"

scenario1_peers.append(start_peer("showcase_coder", "nb_s1_coder"))
print("coder gestartet, warte auf :8802 ...")
assert wait_for_port(8802), "coder kam nicht hoch — siehe examples/acp_showcase/logs/nb_s1_coder.err"

print("\nBeide Peers online:")
free_ports([8801, 8802])

In [ ]:
# Mission durch den Orchestrator laufen lassen. Dauert ~3–6 Min mit gpt-5.4-mini.
MISSION_S1 = (
    "Recherchiere die drei meistgenutzten Python-HTTP-Client-Libraries in 2026 "
    "und schreibe ein minimales lauffaehiges Beispiel mit der empfohlenen Library. "
    "Speichere das Beispiel unter examples/acp_showcase/out/nb_demo_client.py."
)

result = subprocess.run(
    ["uv", "run", "taskforce", "run", "mission",
     "--profile", "showcase_orchestrator", MISSION_S1],
    cwd=str(REPO_ROOT),
    env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print("exit code:", result.returncode)
# Nur die Tail-Sektion mit dem Final Answer + Token Usage anzeigen.
tail = result.stdout.splitlines()[-60:] if result.stdout else []
print("\n".join(tail))
if result.returncode != 0:
    print("\nSTDERR (tail):")
    print("\n".join((result.stderr or "").splitlines()[-30:]))

In [ ]:
# Verifikation: wurde wirklich an die Peers delegiert?
out_file = REPO_ROOT / "examples" / "acp_showcase" / "out" / "nb_demo_client.py"
print("=== Generierte Datei ===")
if out_file.exists():
    print(f"{out_file.relative_to(REPO_ROOT)} ({out_file.stat().st_size} bytes)")
    print(out_file.read_text(encoding="utf-8"))
else:
    print("(noch nicht erzeugt)")

print("\n=== Peer-Aktivitaet (proof of delegation) ===")
for name in ("nb_s1_researcher", "nb_s1_coder"):
    err_path = LOG_DIR / f"{name}.err"
    runs = 0
    if err_path.exists():
        runs = err_path.read_text(encoding="utf-8", errors="replace").count("Run started")
    print(f"  {name:20s} Run-Started-Markers: {runs}")

In [ ]:
# Aufraeumen: Peers killen, Ports freigeben.
for p in scenario1_peers:
    kill_proc_tree(p)
scenario1_peers.clear()
time.sleep(1)
free_ports([8800, 8801, 8802])

## 2. Szenario 2 — Butler delegiert genau einmal

`showcase_butler.yaml` baut einen `native_react`-Agent mit Butler-Tools (`ask_user`, `wiki`, `file_*`) **plus** `call_acp_agent`. Sein `system_prompt` ist hart: "du hast keinen Web-Zugang, fuer alles Aktuelle musst du an `researcher` delegieren".

Wir starten dafuer nur den Researcher-Peer und schicken eine einzelne Frage an den Butler.

> **Hinweis:** Wenn die Web-Suchen vom Researcher gerade massiv blockiert werden (Captchas, 403s) kann es passieren, dass der Butler ueber `ask_user` zurueckfragt statt direkt zu antworten. Das ist trotzdem ein valider Beweis fuer die ACP-Delegation — siehe `pending_question` im persistenten State.

In [ ]:
scenario2_peers: list[subprocess.Popen] = []

scenario2_peers.append(start_peer("showcase_researcher", "nb_s2_researcher"))
print("researcher gestartet, warte auf :8801 ...")
assert wait_for_port(8801), "researcher kam nicht hoch — siehe logs/nb_s2_researcher.err"
print(":8801 OK")

In [ ]:
MISSION_S2 = (
    "Was sind die meistgenutzten Python-Webframeworks im Jahr 2026? "
    "Antworte kurz auf Deutsch und nenne fuer jedes Framework eine Quelle."
)

result = subprocess.run(
    ["uv", "run", "taskforce", "run", "mission",
     "--profile", "showcase_butler", MISSION_S2],
    cwd=str(REPO_ROOT),
    env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print("exit code:", result.returncode)
tail = result.stdout.splitlines()[-50:] if result.stdout else []
print("\n".join(tail))

In [ ]:
# Beweis der Delegation: Researcher-Peer-Log auf 'Run started'/'Run completed' pruefen.
err_path = LOG_DIR / "nb_s2_researcher.err"
if err_path.exists():
    txt = err_path.read_text(encoding="utf-8", errors="replace")
    started = txt.count("Run started")
    completed = txt.count("Run completed")
    print(f"  Run started  : {started}")
    print(f"  Run completed: {completed}")
else:
    print("(kein researcher.err — Peer wurde evtl. nie aufgerufen)")

# Falls Butler ueber ask_user gestoppt hat: pending_question aus dem State zeigen.
state_dir = REPO_ROOT / ".taskforce" / "showcase_butler" / "states"
if state_dir.exists():
    files = sorted(state_dir.glob("*.json"), key=lambda p: p.stat().st_mtime)
    if files:
        d = json.loads(files[-1].read_text(encoding="utf-8"))
        pq = d.get("state_data", {}).get("pending_question")
        if pq:
            print("\nButler hat ueber ask_user gestoppt:")
            print("  Frage    :", pq.get("question"))
            print("  Missing  :", pq.get("missing"))

In [ ]:
# Aufraeumen.
for p in scenario2_peers:
    kill_proc_tree(p)
scenario2_peers.clear()
time.sleep(1)
free_ports([8801])

## 3. Szenario 3 — Message Bus ueber ACP, in-process

Das Aequivalent zu `tests/integration/test_acp_message_bus_e2e.py` — bootet zwei `AcpRuntime`s auf loopback-Ports im **selben Notebook-Prozess**. Kein LLM, kein Subprozess, ein paar Sekunden Laufzeit.

Wichtiger Stolperstein, der hier vermieden wird: `AcpMessageBus.subscribe(topic)` ist ein Async-Generator, der die Inbox-Agent-Registration erst beim ersten `await` durchfuehrt. Sobald der ACP-Server laeuft, lehnt `AcpServer.register_agent` neue Eintraege ab. Deshalb registrieren wir das Inbox-Agent **vor** `runtime.start()` explizit ueber das interne `_ensure_registered` — exakt wie es `application/acp_service.build_message_bus()` fuer auto-konfigurierte `subscribe_topics` macht.

In [ ]:
import asyncio
from contextlib import closing

from taskforce.core.domain.acp import AcpPeer
from taskforce.infrastructure.acp.acp_message_bus import AcpMessageBus
from taskforce.infrastructure.acp.runtime import AcpRuntime

def free_port() -> int:
    with closing(socket.socket(socket.AF_INET, socket.SOCK_STREAM)) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

port_a = free_port()
port_b = free_port()

runtime_a = AcpRuntime(host="127.0.0.1", port=port_a)
runtime_b = AcpRuntime(host="127.0.0.1", port=port_b)

runtime_a.peers.register(AcpPeer(
    name="subscriber",
    base_url=f"http://127.0.0.1:{port_b}",
    agent="bus_demo_events",   # wird vom Bus pro Call ueberschrieben
))

bus_a = AcpMessageBus(runtime_a, publish_peers=["subscriber"])
bus_b = AcpMessageBus(runtime_b)

# Inbox-Agent VOR start() registrieren (sonst RuntimeError).
topic_queue = bus_b._queues.setdefault("demo_events", asyncio.Queue())
bus_b._ensure_registered("demo_events", topic_queue)

await runtime_a.start()
await runtime_b.start()
print(f"runtime A laeuft auf :{port_a}, runtime B auf :{port_b}")

In [ ]:
# Publish auf A -> Subscribe auf B (ueber ACP HTTP).
try:
    iterator_b = bus_b.subscribe("demo_events")

    async def collect():
        async for envelope in iterator_b:
            return envelope

    consumer = asyncio.create_task(collect())
    await asyncio.sleep(0)   # consumer-Task soll erst parken

    published = await bus_a.publish("demo_events", {"hello": "world", "from": "notebook"})
    received = await asyncio.wait_for(consumer, timeout=10.0)

    print("published.message_id =", published.message_id)
    print("received.message_id  =", received.message_id)
    print("received.payload     =", received.payload)
    print("received.topic       =", received.topic)
    assert received.message_id == published.message_id
    assert received.payload == {"hello": "world", "from": "notebook"}
    print("\nOK — Envelope per ACP-Run von A nach B ausgeliefert.")
finally:
    await runtime_a.stop()
    await runtime_b.stop()
    print("runtimes gestoppt.")

## Fertig

Was wir gezeigt haben:

- **Szenario 1** beweist Multi-Step-Orchestrierung ueber `call_acp_agent` — Verifikation per Log-Marker (`Run started`/`Run completed`) und dem von `coder` erzeugten Artefakt unter `examples/acp_showcase/out/nb_demo_client.py`.
- **Szenario 2** zeigt den realistischen Single-Delegation-Fall fuer einen chat-artigen Butler. Selbst wenn die Live-Recherche scheitert (Captchas/403s), bleibt die Delegation nachweisbar, und der Butler eskaliert sauber per `ask_user`.
- **Szenario 3** validiert den Message-Bus end-to-end ueber echte ACP-HTTP-Runs (zwei lokale `AcpRuntime`s im selben Prozess) — der gleiche Pfad, den die Integration-Suite testet.

Weiterfuehrend:

- [`docs/features/acp.md`](../docs/features/acp.md) — Feature-Guide
- [`docs/spec/acp.md`](../docs/spec/acp.md) — Vertrag/Invariants
- [`docs/adr/adr-018-acp-protocol-support.md`](../docs/adr/adr-018-acp-protocol-support.md) — Design-Rationale
- [`tests/integration/test_acp_message_bus_e2e.py`](../tests/integration/test_acp_message_bus_e2e.py) — pytest-Variante von Szenario 3